# 参数敏感性分析

对均值回归策略做网格搜索，观察参数变化对绩效指标的影响。

方法：
1. 定义参数空间（window × num_std）
2. 遍历所有组合，每组跑完整回测管道
3. 画热力图观察参数鲁棒性
4. 选出最优参数并分析

In [ ]:
import sys

sys.path.insert(0, '..')

import matplotlib
import pandas as pd

matplotlib.use('Agg')
from pathlib import Path

import matplotlib.pyplot as plt
from strategies.mean_reversion import mean_reversion_signal

from analysis.param_sweep import best_result, run_sweep
from analysis.plot import plot_sweep_heatmap
from config.loader import load_config
from data.fetcher import fetch_daily
from data.filters import detect_limit_price, detect_suspension
from data.storage import load_parquet, save_parquet
from data.universe import resolve_universe

print('All imports OK')

## Step 1: 加载数据

In [ ]:
cfg = load_config(Path('../configs/default.yaml'))
universe_cfg = cfg.get('universe', {})

STOCKS = resolve_universe(universe_cfg)
START = universe_cfg.get('start_date', '2023-01-01')
END = universe_cfg.get('end_date', '2024-01-01')
RAW_DIR = Path('../data/raw')

frames = []
for code in STOCKS:
    path = RAW_DIR / f'{code}.parquet'
    if path.exists():
        df = load_parquet(path)
    else:
        df = fetch_daily(code, START, END)
        save_parquet(df, path)
    frames.append(df)

data = pd.concat(frames, ignore_index=True)

# 标注市场状态
data = detect_limit_price(data)
data = detect_suspension(data)

print(f'Data: {len(data)} rows, {data["code"].nunique()} stocks')
print(f'Date range: {data["date"].min().date()} ~ {data["date"].max().date()}')

## Step 2: 定义参数空间

In [ ]:
param_grid = {
    'window': [5, 10, 15, 20, 25, 30, 40, 50],
    'num_std': [1.0, 1.5, 2.0, 2.5, 3.0],
}

n_combos = 1
for v in param_grid.values():
    n_combos *= len(v)
print(f'Parameter combinations: {n_combos}')

## Step 3: 运行网格搜索

In [ ]:
results = run_sweep(
    signal_gen=mean_reversion_signal,
    param_grid=param_grid,
    data=data,
    capital=1_000_000,
)

print(f'Results: {len(results)} rows')
results.sort_values('sharpe_ratio', ascending=False).head(10)

## Step 4: 热力图分析

In [ ]:
output_dir = Path('output')
output_dir.mkdir(exist_ok=True)

# Sharpe 热力图
fig = plot_sweep_heatmap(results, 'window', 'num_std', 'sharpe_ratio',
                         title='Mean Reversion — Sharpe Ratio')
fig.savefig(output_dir / 'sweep_sharpe_heatmap.png', dpi=150, bbox_inches='tight')
print('Saved: sweep_sharpe_heatmap.png')
plt.close(fig)

# 总收益率热力图
fig = plot_sweep_heatmap(results, 'window', 'num_std', 'total_return',
                         title='Mean Reversion — Total Return')
fig.savefig(output_dir / 'sweep_return_heatmap.png', dpi=150, bbox_inches='tight')
print('Saved: sweep_return_heatmap.png')
plt.close(fig)

# 最大回撤热力图
fig = plot_sweep_heatmap(results, 'window', 'num_std', 'max_drawdown',
                         title='Mean Reversion — Max Drawdown')
fig.savefig(output_dir / 'sweep_drawdown_heatmap.png', dpi=150, bbox_inches='tight')
print('Saved: sweep_drawdown_heatmap.png')
plt.close(fig)

## Step 5: 最优参数分析

In [ ]:
best = best_result(results)
print('=== Best Parameters (by Sharpe) ===')
print(f'  window  = {best["window"]}')
print(f'  num_std = {best["num_std"]}')
print()
print('=== Best Metrics ===')
for k in ['total_return', 'annual_return', 'sharpe_ratio', 'max_drawdown', 'win_rate', 'trade_count']:
    v = best[k]
    print(f'  {k}: {v:.4f}' if isinstance(v, float) else f'  {k}: {v}')

## Step 6: 参数鲁棒性评估

统计在所有参数组合中，盈利组合的占比。占比越高说明策略越鲁棒。

In [ ]:
total = len(results)
profitable = (results['total_return'] > 0).sum()
good_sharpe = (results['sharpe_ratio'] > 0.5).sum()
good_sharpe_low_dd = ((results['sharpe_ratio'] > 0.5) & (results['max_drawdown'] < 0.15)).sum()

print(f'Total combinations: {total}')
print(f'Profitable (return > 0): {profitable} ({profitable/total*100:.1f}%)')
print(f'Good Sharpe (> 0.5): {good_sharpe} ({good_sharpe/total*100:.1f}%)')
print(f'Good Sharpe + Low DD (< 15%): {good_sharpe_low_dd} ({good_sharpe_low_dd/total*100:.1f}%)')
print()
if profitable / total > 0.6:
    print('Result: 参数高原面积较大，策略有一定鲁棒性')
elif profitable / total > 0.3:
    print('Result: 参数敏感性中等，需要进一步验证')
else:
    print('Result: 参数敏感性高，可能过拟合')

## Step 7: 结果分析

（运行后填写观察）